In [13]:
import os
!pip install segmentation-models
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd
import cv2

from google.colab import drive
drive.mount('/content/drive')

os.environ["SM_FRAMEWORK"] = "tf.keras"
import segmentation_models as sm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
IMAGE_DIR   = "/content/drive/MyDrive/floodsegnet/data/Image"
MASK_DIR    = "/content/drive/MyDrive/floodsegnet/data/Mask"
METADATA    = "/content/drive/MyDrive/floodsegnet/data/metadata.csv"

metadata = pd.read_csv(METADATA)
metadata.head()

image_paths = [
    os.path.join(IMAGE_DIR, img_name)
    for img_name in metadata["Image"]
]

mask_paths = [
    os.path.join(MASK_DIR, mask_name)
    for mask_name in metadata["Mask"]
]

print(f"Images: {len(image_paths)}, Masks: {len(mask_paths)}")

Images: 290, Masks: 290


In [15]:
IMG_SIZE = 256
BATCH_SIZE = 8

In [16]:
def load_single(img_path, mask_path):
    # load a single image
    # read, convert to rgb [opencv reads in bgr], resize, normalise to [0,1] float32

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image.astype(np.float32) / 255.0

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    mask = (mask > 127).astype(np.float32)
    mask = np.expand_dims(mask, axis=-1)

    return image, mask

# load everything into memory upfront — dataset is only 290 images, fits in RAM
all_images = []
all_masks  = []

for img_p, msk_p in zip(image_paths, mask_paths):
    img, msk = load_single(img_p, msk_p)
    all_images.append(img)
    all_masks.append(msk)

all_images = np.array(all_images)  # (290, 256, 256, 3)
all_masks  = np.array(all_masks)   # (290, 256, 256, 1)

print(f"Images: {all_images.shape}")
print(f"Masks:  {all_masks.shape}")

Images: (290, 256, 256, 3)
Masks:  (290, 256, 256, 1)


In [17]:
# split
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    all_images, all_masks,
    test_size=0.2,
    random_state=42
)

In [18]:
# tf datasets — no map, no numpy_function, no AutoGraph issues
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.shuffle(100).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [19]:
# sanity check
for images, masks in train_dataset.take(1):
    print(f"Image batch: {images.shape}")  # (8, 256, 256, 3)
    print(f"Mask batch:  {masks.shape}")   # (8, 256, 256, 1)
    print(f"Mask unique: {np.unique(masks.numpy())}")  # [0. 1.]

Image batch: (8, 256, 256, 3)
Mask batch:  (8, 256, 256, 1)
Mask unique: [0. 1.]


In [20]:
#the fun part
model = sm.Unet(
    backbone_name="resnet34",
    encoder_weights="imagenet",
    classes=1,
    activation="sigmoid"
)

In [21]:
model.compile(
    optimizer='adam',
    loss=sm.losses.bce_dice_loss,
    metrics=[
        sm.metrics.iou_score,
        sm.metrics.f1_score
    ]
)

In [26]:
callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_f1-score",
        mode="max",
        save_best_only=True
    )

]

In [27]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=callbacks
)

BATCH_SIZE = 8

Epoch 1/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 7s 252ms/step - f1-score: 0.8718 - iou_score: 0.7742 - loss: 0.3475 - val_f1-score: 0.5802 - val_iou_score: 0.4112 - val_loss: 4.2241
Epoch 2/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 7s 242ms/step - f1-score: 0.8572 - iou_score: 0.7511 - loss: 0.3820 - val_f1-score: 0.5871 - val_iou_score: 0.4168 - val_loss: 2.3883
Epoch 3/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 5s 178ms/step - f1-score: 0.8606 - iou_score: 0.7565 - loss: 0.3670 - val_f1-score: 0.5775 - val_iou_score: 0.4074 - val_loss: 1.6493
Epoch 4/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 5s 182ms/step - f1-score: 0.8722 - iou_score: 0.7749 - loss: 0.3397 - val_f1-score: 0.5834 - val_iou_score: 0.4142 - val_loss: 3.0900
Epoch 5/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 5s 179ms/step - f1-score: 0.8784 - iou_score: 0.7842 - loss: 0.3147 - val_f1-score: 0.5802 - val_iou_score: 0.4113 - val_loss: 4.3390
Epoch 6/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 5s 181ms/step - f1-score: 0.8859 - iou_score: 0.7962 - loss: 0.3015 - val_f1-score: 0.5863 - val_iou_s

In [24]:
model.save("/content/drive/MyDrive/floodsegnet/best_model_colab.keras")
print("Model saved!")

Model saved!
